In [ ]:
!cp "/content/drive/MyDrive/ecthr_train_pyserini_bm25_top50Precedents - Copy.csv" \
    "/content/drive/MyDrive/ecthr_dev_pyserini_bm25_top50Precedents - Copy.csv" \
    "/content/drive/MyDrive/ecthr_test_pyserini_bm25_top50Precedents - Copy.csv" \
    "/content/"

In [ ]:

import os, random, time, ast
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from google.colab import drive
drive.mount('/content/drive')

torch.backends.cudnn.benchmark = True
device = torch.device("cuda")

class Config:
    # Paths
    TRAIN_PATH = "/content/ecthr_train_pyserini_bm25_top50Precedents - Copy.csv"
    DEV_PATH = "/content/ecthr_dev_pyserini_bm25_top50Precedents - Copy.csv"
    TEST_PATH = "/content/ecthr_test_pyserini_bm25_top50Precedents - Copy.csv"
    OUT_DIR = "/content/drive/MyDrive/Biencoder_Output_Final"

    # Model
    MODEL_NAME = "allenai/longformer-base-4096"
    MAX_LEN = 4096  # 4096 takes 7+ hours, 1024 takes 1.5 hours

    # Training
    BATCH_SIZE = 8
    EPOCHS = 3  # 3 epochs for better accuracy
    LR = 2e-5
    TEMP = 0.07
    NEG_N = 7
    GRAD_CLIP = 1.0
    USE_FP16 = True
    SAVE_EVERY = 500
    NUM_WORKERS = 0

os.makedirs(Config.OUT_DIR, exist_ok=True)

# Helper functions
def safe_str(x):
    if x is None or pd.isna(x):
        return ""
    return str(x)

def parse_citations(raw):
    if raw is None or pd.isna(raw):
        return []
    try:
        if isinstance(raw, str) and raw.startswith('['):
            return [str(x).strip() for x in ast.literal_eval(raw)]
        else:
            return [str(raw).strip()]
    except:
        return [str(raw).strip()] if str(raw).strip() else []

def pick_first(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

# Load data
print("Loading data...")
train_df = pd.read_csv(Config.TRAIN_PATH)
dev_df = pd.read_csv(Config.DEV_PATH)
test_df = pd.read_csv(Config.TEST_PATH)

for df in [train_df, dev_df, test_df]:
    df["appno"] = df["appno"].astype(str)

# Build corpus
corpus_df = pd.concat([train_df, dev_df], ignore_index=True).drop_duplicates(subset=["appno"]).reset_index(drop=True)
appno_to_idx = {a: i for i, a in enumerate(corpus_df["appno"].tolist())}

# Detect columns
FACTS_COL = pick_first(train_df, ["facts", "Facts"])
CITATIONS_COL = pick_first(train_df, ["citations", "gold"])
CONCEPTS_COL = pick_first(train_df, ["oracle_concepts", "concepts", "generated_concepts"])
CORPUS_COLS = [c for c in ["facts", "law", "reasoning"] if c in corpus_df.columns]

print(f"Train: {len(train_df):,} | Corpus: {len(corpus_df):,}")

# Build text stores
print("\nBuilding text stores...")
train_query_texts = []
train_gold_citations = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Queries"):
    c = safe_str(row.get(CONCEPTS_COL, "")) if CONCEPTS_COL else ""
    f = safe_str(row[FACTS_COL])
    query = f"{c} ; {f}".strip() if c else f
    train_query_texts.append(query)

    gold = parse_citations(row[CITATIONS_COL])
    gold = [a for a in gold if a in appno_to_idx]
    train_gold_citations.append(gold)

corpus_doc_texts = []
for _, row in tqdm(corpus_df.iterrows(), total=len(corpus_df), desc="Docs"):
    parts = [safe_str(row[c]) for c in CORPUS_COLS]
    corpus_doc_texts.append("\n".join([p for p in parts if p.strip()]))

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
pad_id = tokenizer.pad_token_id

# Pre-tokenize corpus
print("\nPre-tokenizing corpus...")
corpus_tokens = []
for text in tqdm(corpus_doc_texts, desc="Tokenizing"):
    if not text.strip():
        text = " "
    tokens = tokenizer(text, truncation=True, max_length=Config.MAX_LEN, padding=False, return_tensors="pt")
    corpus_tokens.append({
        "input_ids": tokens["input_ids"][0],
        "attention_mask": tokens["attention_mask"][0]
    })

# Model
class BiEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(Config.MODEL_NAME, ignore_mismatched_sizes=True)
        self.drop = nn.Dropout(0.1)

    def encode(self, input_ids, attention_mask):
        gam = torch.zeros_like(input_ids)
        gam[:, 0] = 1
        out = self.enc(input_ids=input_ids, attention_mask=attention_mask,
                       global_attention_mask=gam)
        cls = out.last_hidden_state[:, 0, :]
        return F.normalize(self.drop(cls), dim=-1)

model = BiEncoder().to(device)

# Pre-encode corpus
print("\nPre-encoding corpus...")
model.eval()
corpus_embs = []
with torch.no_grad():
    for i in tqdm(range(0, len(corpus_tokens), 32), desc="Encoding"):
        batch = corpus_tokens[i:i+32]
        max_len = max(t["input_ids"].size(0) for t in batch)
        max_len = ((max_len + 511) // 512) * 512

        ids = torch.stack([F.pad(t["input_ids"], (0, max_len - t["input_ids"].size(0)), value=pad_id) for t in batch])
        mask = torch.stack([F.pad(t["attention_mask"], (0, max_len - t["attention_mask"].size(0)), value=0) for t in batch])

        embs = model.encode(ids.to(device), mask.to(device))
        corpus_embs.append(embs.cpu())

corpus_embs = torch.cat(corpus_embs).to(device)
print(f"Corpus encoded: {corpus_embs.shape}")

# Dataset
class RetrievalDataset(Dataset):
    def __init__(self, query_texts, gold_citations):
        self.qtexts = query_texts
        self.gold = gold_citations

    def __len__(self):
        return len(self.qtexts)

    def __getitem__(self, idx):
        gold = self.gold[idx]
        if not gold:
            return None

        pos_app = random.choice(gold)
        pos_idx = appno_to_idx[pos_app]

        neg_idxs = []
        exclude = set(gold)
        while len(neg_idxs) < Config.NEG_N:
            j = random.randrange(len(corpus_df))
            app_j = str(corpus_df.iloc[j]["appno"])
            if app_j not in exclude:
                neg_idxs.append(j)
                exclude.add(app_j)

        q_tok = tokenizer(
            self.qtexts[idx],
            truncation=True,
            max_length=Config.MAX_LEN,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "q_ids": q_tok["input_ids"][0],
            "q_mask": q_tok["attention_mask"][0],
            "pos_idx": torch.tensor(pos_idx, dtype=torch.long),
            "neg_idxs": torch.tensor(neg_idxs, dtype=torch.long)
        }

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    return {
        "q_ids": torch.stack([b["q_ids"] for b in batch]),
        "q_mask": torch.stack([b["q_mask"] for b in batch]),
        "pos_idxs": torch.stack([b["pos_idx"] for b in batch]),
        "neg_idxs": torch.stack([b["neg_idxs"] for b in batch])
    }

# Create dataset
train_dataset = RetrievalDataset(train_query_texts, train_gold_citations)
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

# Count valid samples
valid_count = sum(1 for i in range(min(1000, len(train_dataset))) if train_dataset[i] is not None)
print(f"Valid samples: ~{valid_count}/1000")

# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=0.01)
total_steps = len(train_loader) * Config.EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=Config.USE_FP16)

# Resume checkpoint
CKPT_PATH = os.path.join(Config.OUT_DIR, "checkpoint.pt")
start_epoch = 0
start_step = 0

if os.path.exists(CKPT_PATH):
    print(f"Loading checkpoint from {CKPT_PATH}...")
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt.get('epoch', 0)
        start_step = ckpt.get('step', 0)
        print(f"Resumed from epoch {start_epoch}, step {start_step}")
    except Exception as e:
        print(f"Error loading checkpoint: {e}")

def save_checkpoint(epoch, step):
    torch.save({
        'epoch': epoch,
        'step': step,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
    }, CKPT_PATH)

# Training
print(f"\n{'='*60}")
print("TRAINING STARTING")
print(f"{'='*60}")
print(f"Batch size: {Config.BATCH_SIZE}")
print(f"Total batches: {len(train_loader):,}")
print(f"Total steps: {total_steps:,}")
print(f"Epochs: {Config.EPOCHS}")
print(f"{'='*60}\n")

model.train()
running_loss = 0.0
global_step = 0
t0 = time.time()

for epoch in range(start_epoch, Config.EPOCHS):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        # Skip already processed batches
        if epoch == start_epoch and global_step < start_step:
            global_step += 1
            continue

        # Move to GPU
        q_ids = batch["q_ids"].to(device)
        q_mask = batch["q_mask"].to(device)
        pos_idxs = batch["pos_idxs"].to(device)
        neg_idxs = batch["neg_idxs"].to(device)

        batch_size = q_ids.shape[0]

        with torch.amp.autocast("cuda", enabled=Config.USE_FP16):
            q_emb = model.encode(q_ids, q_mask)
            pos_embs = corpus_embs[pos_idxs]

            neg_flat = neg_idxs.view(-1)
            neg_embs_flat = corpus_embs[neg_flat]
            neg_embs = neg_embs_flat.view(batch_size, Config.NEG_N, -1)

            all_doc_embs = torch.cat([pos_embs.unsqueeze(1), neg_embs], dim=1)
            scores = torch.bmm(q_emb.unsqueeze(1), all_doc_embs.transpose(1, 2)).squeeze(1) / Config.TEMP
            labels = torch.zeros(batch_size, dtype=torch.long, device=device)
            loss = F.cross_entropy(scores, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item()
        global_step += 1

        # Update progress
        avg_loss = running_loss / global_step
        elapsed = (time.time() - t0) / 60
        eta = (total_steps - global_step) * (elapsed / max(global_step, 1))

        pbar.set_postfix(
            loss=f"{avg_loss:.4f}",
            step=global_step,
            eta=f"{eta:.0f}m"
        )

        # Save checkpoint
        if global_step % Config.SAVE_EVERY == 0:
            save_checkpoint(epoch, global_step)
            print(f"\n  ✓ Checkpoint saved at step {global_step}")

# Final save
final_path = os.path.join(Config.OUT_DIR, "model_final.pt")
torch.save(model.state_dict(), final_path)
total_time = (time.time() - t0) / 60

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"Total time: {total_time:.1f} minutes")
print(f"Final loss: {running_loss/global_step:.4f}")
print(f"Model saved: {final_path}")
print(f"{'='*60}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...
Train: 10,284 | Corpus: 12,480

Building text stores...


Docs: 100%|██████████| 12480/12480 [00:01<00:00, 11499.82it/s]



Pre-tokenizing corpus...


Tokenizing: 100%|██████████| 12480/12480 [06:17<00:00, 33.05it/s]


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

...

Epoch 3/3:  33%|███▎      | 428/1286 [11:34<2:40:55, 11.25s/it, eta=23m, loss=1.7773, step=3000]


  ✓ Checkpoint saved at step 3000


Epoch 3/3:  72%|███████▏  | 928/1286 [24:52<45:43,  7.66s/it, eta=10m, loss=1.7654, step=3500]


  ✓ Checkpoint saved at step 3500


Epoch 3/3:  99%|█████████▊| 1269/1286 [33:47<00:25,  1.52s/it, eta=0m, loss=1.7586, step=3841]

In [ ]:
# ============================================================
# EVALUATION CODE
# ============================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

# ============================================================
# 1. LOAD THE TRAINED MODEL
# ============================================================

print("="*60)
print("LOADING TRAINED MODEL")
print("="*60)

# Path to your trained model
MODEL_PATH = "/content/drive/MyDrive/Biencoder_Output_Final/model_final.pt"
OUT_DIR = "/content/drive/MyDrive/Biencoder_Output_Final"

# Load the model architecture
class BiEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained("allenai/longformer-base-4096", ignore_mismatched_sizes=True)
        self.drop = nn.Dropout(0.1)

    def encode(self, input_ids, attention_mask):
        gam = torch.zeros_like(input_ids)
        gam[:, 0] = 1
        out = self.enc(input_ids=input_ids, attention_mask=attention_mask,
                       global_attention_mask=gam)
        cls = out.last_hidden_state[:, 0, :]
        return F.normalize(self.drop(cls), dim=-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiEncoder().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"✓ Model loaded from {MODEL_PATH}")

# ============================================================
# 2. LOAD TOKENIZER AND DATA
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
pad_id = tokenizer.pad_token_id

# Load test data
TEST_PATH = "/content/ecthr_test_pyserini_bm25_top50Precedents - Copy.csv"
test_df = pd.read_csv(TEST_PATH)
test_df["appno"] = test_df["appno"].astype(str)

# Load corpus (same as training)
train_df = pd.read_csv("/content/ecthr_train_pyserini_bm25_top50Precedents - Copy.csv")
dev_df = pd.read_csv("/content/ecthr_dev_pyserini_bm25_top50Precedents - Copy.csv")
corpus_df = pd.concat([train_df, dev_df], ignore_index=True).drop_duplicates(subset=["appno"]).reset_index(drop=True)

# Create mapping
appno_to_idx = {a: i for i, a in enumerate(corpus_df["appno"].tolist())}
idx_to_appno = {i: a for a, i in appno_to_idx.items()}

# Detect columns
def pick_first(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

FACTS_COL = pick_first(test_df, ["facts", "Facts"])
CITATIONS_COL = pick_first(test_df, ["citations", "gold"])
CONCEPTS_COL = pick_first(test_df, ["oracle_concepts", "concepts", "generated_concepts"])

print(f"✓ Data loaded: {len(test_df):,} test queries, {len(corpus_df):,} corpus docs")

# ============================================================
# 3. BUILD CORPUS EMBEDDINGS (if not already cached)
# ============================================================

CORPUS_EMBS_PATH = os.path.join(OUT_DIR, "corpus_embeddings.pt")

if os.path.exists(CORPUS_EMBS_PATH):
    print("Loading cached corpus embeddings...")
    corpus_embs = torch.load(CORPUS_EMBS_PATH, map_location=device)
    print(f"✓ Loaded corpus embeddings: {corpus_embs.shape}")
else:
    print("Building corpus embeddings (one-time, ~5-10 minutes)...")

    # Build corpus texts
    CORPUS_COLS = [c for c in ["facts", "law", "reasoning"] if c in corpus_df.columns]
    corpus_texts = []
    for _, row in tqdm(corpus_df.iterrows(), total=len(corpus_df), desc="Building corpus texts"):
        parts = []
        for col in CORPUS_COLS:
            if pd.notna(row[col]):
                parts.append(str(row[col]))
        corpus_texts.append("\n".join(parts))

    # Encode in batches
    corpus_embs = []
    batch_size = 32

    with torch.no_grad():
        for i in tqdm(range(0, len(corpus_texts), batch_size), desc="Encoding corpus"):
            batch_texts = corpus_texts[i:i+batch_size]

            # Tokenize batch
            tokens = tokenizer(batch_texts, truncation=True, max_length=1024,
                              padding=True, return_tensors="pt")

            # Move to device
            input_ids = tokens["input_ids"].to(device)
            attention_mask = tokens["attention_mask"].to(device)

            # Encode
            embs = model.encode(input_ids, attention_mask)
            corpus_embs.append(embs.cpu())

    corpus_embs = torch.cat(corpus_embs, dim=0)
    torch.save(corpus_embs, CORPUS_EMBS_PATH)
    corpus_embs = corpus_embs.to(device)
    print(f"✓ Corpus embeddings saved: {corpus_embs.shape}")

# ============================================================
# 4. EVALUATION FUNCTION
# ============================================================

def parse_citations(raw):
    """Parse citations from various formats"""
    if raw is None or pd.isna(raw):
        return []
    import ast
    try:
        if isinstance(raw, str) and raw.startswith('['):
            return [str(x).strip() for x in ast.literal_eval(raw)]
        else:
            return [str(raw).strip()]
    except:
        return [str(raw).strip()] if str(raw).strip() else []

def evaluate_recall(test_df, corpus_embs, k_values=[50, 100, 500], max_queries=None):
    """Evaluate recall@k for test queries"""

    recalls = {k: [] for k in k_values}
    skipped = 0
    query_texts = []

    # Build query texts
    for _, row in test_df.iterrows():
        if CONCEPTS_COL and pd.notna(row.get(CONCEPTS_COL)):
            c = str(row[CONCEPTS_COL])
            f = str(row[FACTS_COL]) if FACTS_COL else ""
            query = f"{c} ; {f}".strip()
        else:
            query = str(row[FACTS_COL]) if FACTS_COL else ""
        query_texts.append(query)

    # Limit queries if specified
    if max_queries:
        query_texts = query_texts[:max_queries]
        test_df = test_df.iloc[:max_queries]

    print(f"\nEvaluating {len(query_texts):,} queries...")

    with torch.no_grad():
        for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating")):
            # Get gold citations
            gold_raw = row[CITATIONS_COL]
            gold = [a for a in parse_citations(gold_raw) if a in appno_to_idx]

            if not gold:
                skipped += 1
                continue

            # Encode query
            tokens = tokenizer(query_texts[idx], truncation=True, max_length=1024,
                              return_tensors="pt")
            input_ids = tokens["input_ids"].to(device)
            attention_mask = tokens["attention_mask"].to(device)

            q_emb = model.encode(input_ids, attention_mask)

            # Compute similarities
            scores = torch.matmul(q_emb, corpus_embs.T).squeeze(0)

            # Get top-k retrievals
            for k in k_values:
                top_k = min(k, len(corpus_embs))
                top_indices = torch.topk(scores, top_k).indices.tolist()
                retrieved = [idx_to_appno[i] for i in top_indices]

                # Calculate recall
                recall = len(set(retrieved) & set(gold)) / len(gold)
                recalls[k].append(recall)

    return recalls, skipped

# ============================================================
# 5. RUN EVALUATION
# ============================================================

print("\n" + "="*60)
print("RUNNING EVALUATION")
print("="*60)

# Evaluate on full test set
recalls, skipped = evaluate_recall(test_df, corpus_embs, k_values=[50, 100, 500])

# Print results
print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Test queries: {len(test_df):,}")
print(f"Skipped (no gold in corpus): {skipped}")
print(f"Valid queries evaluated: {len(recalls[50]):,}")
print("\n" + "-"*60)
print("RECALL@K (True Recall - paper definition)")
print("-"*60)

for k in [50, 100, 500]:
    mean_recall = np.mean(recalls[k]) * 100
    std_recall = np.std(recalls[k]) * 100
    print(f"Recall@{k:3d}:  {mean_recall:6.2f}% (±{std_recall:.2f}%)")

print("-"*60)

# Compare to paper baseline
print("\n📊 COMPARISON TO PAPER (LeCoPCR):")
print(f"  Your Recall@100:  {np.mean(recalls[100])*100:.2f}%")
print(f"  Paper baseline:   33.97% (with 4096 tokens)")
print(f"  Expected (1024):  20-28%")

if np.mean(recalls[100]) >= 0.25:
    print("  ✅ EXCELLENT! Model exceeds expectations.")
elif np.mean(recalls[100]) >= 0.20:
    print("  ✅ GOOD! Model meets expected performance for 1024 tokens.")
elif np.mean(recalls[100]) >= 0.15:
    print("  ⚠️ ACCEPTABLE but below expected. Consider more epochs or 4096 tokens.")
else:
    print("  ❌ BELOW EXPECTED. Check data quality or increase training.")

# ============================================================
# 6. DETAILED ANALYSIS BY QUERY TYPE
# ============================================================

print("\n" + "="*60)
print("DETAILED ANALYSIS")
print("="*60)

# Analyze by number of gold citations
gold_counts = []
for _, row in test_df.iterrows():
    gold = [a for a in parse_citations(row[CITATIONS_COL]) if a in appno_to_idx]
    gold_counts.append(len(gold))

gold_counts = np.array(gold_counts)
recall_100 = np.array(recalls[100])

print("\nPerformance by number of gold citations:")
for num_gold in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    mask = gold_counts == num_gold
    if mask.sum() > 0:
        avg_recall = recall_100[mask].mean() * 100
        print(f"  {num_gold} gold cites ({mask.sum():4d} queries): Recall@100 = {avg_recall:5.2f}%")

# Show example retrievals
print("\n" + "="*60)
print("EXAMPLE RETRIEVALS")
print("="*60)

with torch.no_grad():
    for i in range(min(5, len(test_df))):
        row = test_df.iloc[i]
        gold = [a for a in parse_citations(row[CITATIONS_COL]) if a in appno_to_idx]

        if gold:
            # Encode query
            tokens = tokenizer(query_texts[i], truncation=True, max_length=1024,
                              return_tensors="pt")
            input_ids = tokens["input_ids"].to(device)
            attention_mask = tokens["attention_mask"].to(device)

            q_emb = model.encode(input_ids, attention_mask)
            scores = torch.matmul(q_emb, corpus_embs.T).squeeze(0)
            top10 = torch.topk(scores, 10).indices.tolist()
            retrieved = [idx_to_appno[j] for j in top10]

            print(f"\nQuery {i+1}:")
            print(f"  Text: {query_texts[i][:150]}...")
            print(f"  Gold: {gold[:5]}")
            print(f"  Top-5 retrieved: {retrieved[:5]}")

            # Check if gold found
            found = [g for g in gold if g in retrieved[:100]]
            print(f"  Gold in top-100: {len(found)}/{len(gold)} ({len(found)/len(gold)*100:.0f}%)")

# ============================================================
# 7. SAVE RESULTS
# ============================================================

results = {
    "recall_50": recalls[50],
    "recall_100": recalls[100],
    "recall_500": recalls[500],
    "mean_recall_50": np.mean(recalls[50]),
    "mean_recall_100": np.mean(recalls[100]),
    "mean_recall_500": np.mean(recalls[500]),
    "std_recall_100": np.std(recalls[100]),
    "queries_evaluated": len(recalls[100]),
    "queries_skipped": skipped
}

results_path = os.path.join(OUT_DIR, "evaluation_results.pt")
torch.save(results, results_path)
print(f"\n✓ Results saved to {results_path}")

# Also save as CSV
results_df = pd.DataFrame({
    "recall_50": recalls[50],
    "recall_100": recalls[100],
    "recall_500": recalls[500]
})
results_df.to_csv(os.path.join(OUT_DIR, "evaluation_results.csv"), index=False)
print(f"✓ Results also saved as CSV")

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE!")
print("="*60)

LOADING TRAINED MODEL


Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded from /content/drive/MyDrive/Biencoder_Output_Final/model_final.pt
✓ Data loaded: 2,838 test queries, 12,480 corpus docs
Building corpus embeddings (one-time, ~5-10 minutes)...


Encoding corpus: 100%|██████████| 390/390 [30:06<00:00,  4.63s/it]


✓ Corpus embeddings saved: torch.Size([12480, 768])

RUNNING EVALUATION

Evaluating 2,838 queries...


Evaluating: 100%|██████████| 2838/2838 [06:28<00:00,  7.31it/s]



EVALUATION RESULTS
Test queries: 2,838
Skipped (no gold in corpus): 38
Valid queries evaluated: 2,800

------------------------------------------------------------
RECALL@K (True Recall - paper definition)
------------------------------------------------------------
Recall@ 50:   12.41% (±18.03%)
Recall@100:   17.80% (±20.98%)
Recall@500:   37.62% (±27.68%)
------------------------------------------------------------

📊 COMPARISON TO PAPER (LeCoPCR):
  Your Recall@100:  17.80%
  Paper baseline:   33.97% (with 4096 tokens)
  Expected (1024):  20-28%
  ⚠️ ACCEPTABLE but below expected. Consider more epochs or 4096 tokens.

DETAILED ANALYSIS

Performance by number of gold citations:


IndexError: boolean index did not match indexed array along axis 0; size of axis is 2800 but size of corresponding boolean axis is 2838

In [ ]:
# ============================================================
# PAPER SPECIFICATION BI-ENCODER - LeCoPCR Reproduction
# ============================================================
# Features:
# - 4096 token length (paper default)
# - 3 epochs for optimal convergence
# - Hard negatives from BM25
# - Temperature = 0.07 (paper value)
# - Gradient checkpointing enabled
# - Full checkpoint resume support
# Expected Recall@100: 28-34%
# Training time: 7-8 hours on T4 GPU
# ============================================================

import os, random, time, ast
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from google.colab import drive
drive.mount('/content/drive')

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
device = torch.device("cuda")

class Config:
    # Paths
    TRAIN_PATH = "/content/ecthr_train_pyserini_bm25_top50Precedents - Copy.csv"
    DEV_PATH = "/content/ecthr_dev_pyserini_bm25_top50Precedents - Copy.csv"
    TEST_PATH = "/content/ecthr_test_pyserini_bm25_top50Precedents - Copy.csv"
    OUT_DIR = "/content/drive/MyDrive/Biencoder_Paper_Spec"

    # PAPER SPECIFICATIONS - DO NOT CHANGE
    MODEL_NAME = "allenai/longformer-base-4096"
    MAX_LEN = 4096  # Paper uses 4096 tokens
    BATCH_SIZE = 4  # Reduced due to 4096 token length
    EPOCHS = 3      # Paper uses 3-5 epochs
    LR = 2e-5       # Paper learning rate
    TEMP = 0.07     # Paper temperature
    NEG_N = 7       # Paper uses 7 negatives
    GRAD_ACCUM = 2  # Effective batch size = 4 * 2 = 8

    # Training optimizations
    GRAD_CLIP = 1.0
    USE_FP16 = True
    SAVE_EVERY = 200
    NUM_WORKERS = 2
    USE_HARD_NEGATIVES = True  # Use BM25 top results as hard negatives

os.makedirs(Config.OUT_DIR, exist_ok=True)

# Helper functions
def safe_str(x):
    if x is None or pd.isna(x):
        return ""
    return str(x)

def parse_citations(raw):
    if raw is None or pd.isna(raw):
        return []
    try:
        if isinstance(raw, str) and raw.startswith('['):
            return [str(x).strip() for x in ast.literal_eval(raw)]
        else:
            return [str(raw).strip()]
    except:
        return [str(raw).strip()] if str(raw).strip() else []

def pick_first(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

# ============================================================
# LOAD DATA
# ============================================================
print("="*60)
print("LOADING DATA - Paper Specification")
print("="*60)

train_df = pd.read_csv(Config.TRAIN_PATH)
dev_df = pd.read_csv(Config.DEV_PATH)
test_df = pd.read_csv(Config.TEST_PATH)

for df in [train_df, dev_df, test_df]:
    df["appno"] = df["appno"].astype(str)

# Build corpus (full, no truncation)
corpus_df = pd.concat([train_df, dev_df], ignore_index=True).drop_duplicates(subset=["appno"]).reset_index(drop=True)
appno_to_idx = {a: i for i, a in enumerate(corpus_df["appno"].tolist())}

# Detect columns
FACTS_COL = pick_first(train_df, ["facts", "Facts"])
CITATIONS_COL = pick_first(train_df, ["citations", "gold"])
CONCEPTS_COL = pick_first(train_df, ["oracle_concepts", "concepts", "generated_concepts"])
CORPUS_COLS = [c for c in ["facts", "law", "reasoning"] if c in corpus_df.columns]

print(f"Train: {len(train_df):,} | Corpus: {len(corpus_df):,} | Test: {len(test_df):,}")

# ============================================================
# BUILD TEXT STORES
# ============================================================
print("\nBuilding text stores...")

# Training queries
train_query_texts = []
train_gold_citations = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Queries"):
    c = safe_str(row.get(CONCEPTS_COL, "")) if CONCEPTS_COL else ""
    f = safe_str(row[FACTS_COL])
    query = f"{c} ; {f}".strip() if c else f
    train_query_texts.append(query)

    gold = parse_citations(row[CITATIONS_COL])
    gold = [a for a in gold if a in appno_to_idx]
    train_gold_citations.append(gold)

# Corpus documents
corpus_doc_texts = []
for _, row in tqdm(corpus_df.iterrows(), total=len(corpus_df), desc="Docs"):
    parts = [safe_str(row[c]) for c in CORPUS_COLS]
    corpus_doc_texts.append("\n".join([p for p in parts if p.strip()]))

# ============================================================
# TOKENIZER
# ============================================================
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
pad_id = tokenizer.pad_token_id

# ============================================================
# PRE-TOKENIZE CORPUS (for speed)
# ============================================================
print("\nPre-tokenizing corpus (this takes ~15-20 min for 4096 tokens)...")
corpus_tokens = []
for text in tqdm(corpus_doc_texts, desc="Tokenizing"):
    if not text.strip():
        text = " "
    tokens = tokenizer(text, truncation=True, max_length=Config.MAX_LEN, padding=False, return_tensors="pt")
    corpus_tokens.append({
        "input_ids": tokens["input_ids"][0],
        "attention_mask": tokens["attention_mask"][0]
    })

# ============================================================
# MODEL (with gradient checkpointing for memory efficiency)
# ============================================================
class BiEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(Config.MODEL_NAME, ignore_mismatched_sizes=True)
        self.drop = nn.Dropout(0.1)
        # Enable gradient checkpointing for memory efficiency with 4096 tokens
        self.enc.gradient_checkpointing_enable()
        print("✓ Gradient checkpointing enabled")

    def encode(self, input_ids, attention_mask):
        # Global attention on CLS token only (paper specification)
        gam = torch.zeros_like(input_ids)
        gam[:, 0] = 1
        out = self.enc(input_ids=input_ids, attention_mask=attention_mask,
                       global_attention_mask=gam)
        cls = out.last_hidden_state[:, 0, :]
        return F.normalize(self.drop(cls), dim=-1)

model = BiEncoder().to(device)
print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")

# ============================================================
# PRE-ENCODE CORPUS (one-time, saves hours of training)
# ============================================================
print("\nPre-encoding entire corpus (this takes ~30-40 min for 4096 tokens)...")
model.eval()
corpus_embs = []
enc_batch_size = 4  # Smaller due to 4096 tokens

with torch.no_grad():
    for i in tqdm(range(0, len(corpus_tokens), enc_batch_size), desc="Encoding corpus"):
        batch = corpus_tokens[i:i+enc_batch_size]

        # Find max length in batch
        max_len = max(t["input_ids"].size(0) for t in batch)
        max_len = ((max_len + 511) // 512) * 512  # Pad to multiple of 512

        # Pad batch
        ids_list, mask_list = [], []
        for t in batch:
            ids_padded = F.pad(t["input_ids"], (0, max_len - t["input_ids"].size(0)), value=pad_id)
            mask_padded = F.pad(t["attention_mask"], (0, max_len - t["attention_mask"].size(0)), value=0)
            ids_list.append(ids_padded)
            mask_list.append(mask_padded)

        d_ids = torch.stack(ids_list).to(device)
        d_mask = torch.stack(mask_list).to(device)

        embs = model.encode(d_ids, d_mask)
        corpus_embs.append(embs.cpu())

        if i % 100 == 0:
            torch.cuda.empty_cache()

corpus_embs = torch.cat(corpus_embs, dim=0).to(device)
print(f"✓ Corpus pre-encoded: {corpus_embs.shape}")

# ============================================================
# HARD NEGATIVES FROM BM25 (if available in CSV)
# ============================================================
print("\nPreparing negatives...")

def get_hard_negatives(row, gold_set, num_needed=Config.NEG_N):
    """Extract hard negatives from BM25 top results if available"""
    hard_negatives = []

    # Check if BM25 results column exists
    bm25_col = None
    for col in row.index:
        if 'bm25' in col.lower() or 'top50' in col.lower():
            bm25_col = col
            break

    if bm25_col and Config.USE_HARD_NEGATIVES:
        try:
            bm25_results = parse_citations(row[bm25_col])
            for result in bm25_results:
                if result not in gold_set and result in appno_to_idx:
                    hard_negatives.append(appno_to_idx[result])
                    if len(hard_negatives) >= num_needed:
                        break
        except:
            pass

    return hard_negatives

# ============================================================
# DATASET WITH HARD NEGATIVES
# ============================================================
class PaperDataset(Dataset):
    def __init__(self, query_texts, gold_citations, df):
        self.qtexts = query_texts
        self.gold = gold_citations
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.qtexts)

    def __getitem__(self, idx):
        gold = self.gold[idx]
        if not gold:
            return None

        # Positive sample
        pos_app = random.choice(gold)
        pos_idx = appno_to_idx[pos_app]
        gold_set = set(gold)

        # Get hard negatives from BM25
        row = self.df.iloc[idx]
        hard_negs = get_hard_negatives(row, gold_set)

        # Fill remaining with random negatives
        neg_idxs = hard_negs.copy()
        exclude = gold_set.copy()
        for neg in hard_negs:
            exclude.add(str(corpus_df.iloc[neg]["appno"]))

        while len(neg_idxs) < Config.NEG_N:
            j = random.randrange(len(corpus_df))
            app_j = str(corpus_df.iloc[j]["appno"])
            if app_j not in exclude:
                neg_idxs.append(j)
                exclude.add(app_j)

        # Tokenize query with padding for 4096 length
        q_tok = tokenizer(
            self.qtexts[idx],
            truncation=True,
            max_length=Config.MAX_LEN,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "q_ids": q_tok["input_ids"][0],
            "q_mask": q_tok["attention_mask"][0],
            "pos_idx": torch.tensor(pos_idx, dtype=torch.long),
            "neg_idxs": torch.tensor(neg_idxs, dtype=torch.long)
        }

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    return {
        "q_ids": torch.stack([b["q_ids"] for b in batch]),
        "q_mask": torch.stack([b["q_mask"] for b in batch]),
        "pos_idxs": torch.stack([b["pos_idx"] for b in batch]),
        "neg_idxs": torch.stack([b["neg_idxs"] for b in batch])
    }

# Create dataset
train_dataset = PaperDataset(train_query_texts, train_gold_citations, train_df)
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

# Count valid samples
valid_count = 0
for i in range(min(500, len(train_dataset))):
    if train_dataset[i] is not None:
        valid_count += 1
print(f"Valid samples: ~{valid_count}/500 sampled")

# ============================================================
# TRAINING SETUP
# ============================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=0.01)
total_steps = len(train_loader) * Config.EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=Config.USE_FP16)

# Resume checkpoint
CKPT_PATH = os.path.join(Config.OUT_DIR, "checkpoint.pt")
start_epoch = 0
start_step = 0

if os.path.exists(CKPT_PATH):
    print(f"\nLoading checkpoint from {CKPT_PATH}...")
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt.get('epoch', 0)
        start_step = ckpt.get('step', 0)
        print(f"✓ Resumed from epoch {start_epoch}, step {start_step}")
    except Exception as e:
        print(f"Error loading checkpoint: {e}")

def save_checkpoint(epoch, step):
    torch.save({
        'epoch': epoch,
        'step': step,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
    }, CKPT_PATH)

# ============================================================
# TRAINING LOOP
# ============================================================
print("\n" + "="*60)
print("PAPER SPECIFICATION TRAINING")
print("="*60)
print(f"Max length: {Config.MAX_LEN} tokens (paper: 4096)")
print(f"Batch size: {Config.BATCH_SIZE} (effective: {Config.BATCH_SIZE * Config.GRAD_ACCUM})")
print(f"Epochs: {Config.EPOCHS}")
print(f"Temperature: {Config.TEMP}")
print(f"Hard negatives: {Config.USE_HARD_NEGATIVES}")
print(f"Total batches: {len(train_loader):,}")
print(f"Total steps: {total_steps:,}")
print(f"Expected training time: 7-8 hours")
print(f"{'='*60}\n")

model.train()
running_loss = 0.0
global_step = 0
accum_step = 0
t0 = time.time()

for epoch in range(start_epoch, Config.EPOCHS):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
    for batch in pbar:
        if batch is None:
            continue

        # Skip already processed steps
        if epoch == start_epoch and global_step < start_step:
            global_step += 1
            continue

        # Move to GPU
        q_ids = batch["q_ids"].to(device, non_blocking=True)
        q_mask = batch["q_mask"].to(device, non_blocking=True)
        pos_idxs = batch["pos_idxs"].to(device, non_blocking=True)
        neg_idxs = batch["neg_idxs"].to(device, non_blocking=True)

        batch_size = q_ids.shape[0]

        with torch.amp.autocast("cuda", enabled=Config.USE_FP16):
            # Encode queries
            q_emb = model.encode(q_ids, q_mask)

            # Get positive embeddings
            pos_embs = corpus_embs[pos_idxs]

            # Get negative embeddings (vectorized)
            neg_flat = neg_idxs.view(-1)
            neg_embs_flat = corpus_embs[neg_flat]
            neg_embs = neg_embs_flat.view(batch_size, Config.NEG_N, -1)

            # Combine and compute scores
            all_doc_embs = torch.cat([pos_embs.unsqueeze(1), neg_embs], dim=1)
            scores = torch.bmm(q_emb.unsqueeze(1), all_doc_embs.transpose(1, 2)).squeeze(1)
            scores = scores / Config.TEMP

            labels = torch.zeros(batch_size, dtype=torch.long, device=device)
            loss = F.cross_entropy(scores, labels) / Config.GRAD_ACCUM

        # Backward pass with gradient accumulation
        scaler.scale(loss).backward()
        accum_step += 1

        if accum_step % Config.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            # Update metrics
            running_loss += loss.item() * Config.GRAD_ACCUM
            avg_loss = running_loss / global_step
            elapsed = (time.time() - t0) / 60
            eta = (total_steps - global_step) * (elapsed / max(global_step, 1))

            pbar.set_postfix(
                loss=f"{avg_loss:.4f}",
                step=global_step,
                epoch=f"{epoch+1}/{Config.EPOCHS}",
                eta=f"{eta:.0f}m"
            )

            # Save checkpoint
            if global_step % Config.SAVE_EVERY == 0:
                save_checkpoint(epoch, global_step)
                print(f"\n  ✓ Checkpoint saved at step {global_step}")

    # End of epoch - save model
    epoch_model_path = os.path.join(Config.OUT_DIR, f"model_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), epoch_model_path)
    print(f"\n✓ Epoch {epoch+1} completed! Model saved to {epoch_model_path}")
    print(f"  Average loss: {running_loss/global_step:.4f}")

    # Reset step counter for next epoch
    start_step = 0

# Final save
final_path = os.path.join(Config.OUT_DIR, "model_final_paper_spec.pt")
torch.save(model.state_dict(), final_path)
total_time = (time.time() - t0) / 60

print("\n" + "="*60)
print("✅ PAPER SPECIFICATION TRAINING COMPLETE!")
print("="*60)
print(f"Total training time: {total_time:.1f} minutes ({total_time/60:.1f} hours)")
print(f"Final loss: {running_loss/global_step:.4f}")
print(f"Model saved: {final_path}")
print(f"Checkpoints saved in: {Config.OUT_DIR}")
print("="*60)

# ============================================================
# QUICK EVALUATION AFTER TRAINING
# ============================================================
print("\n" + "="*60)
print("RUNNING QUICK EVALUATION")
print("="*60)

# Quick evaluation on first 500 test queries
model.eval()
test_sample = test_df.head(500)
recalls = []

with torch.no_grad():
    for _, row in tqdm(test_sample.iterrows(), total=len(test_sample), desc="Evaluating"):
        gold = [a for a in parse_citations(row[CITATIONS_COL]) if a in appno_to_idx]
        if not gold:
            continue

        # Build query
        c = safe_str(row.get(CONCEPTS_COL, "")) if CONCEPTS_COL else ""
        f = safe_str(row[FACTS_COL])
        query = f"{c} ; {f}".strip() if c else f

        # Encode
        tokens = tokenizer(query, truncation=True, max_length=Config.MAX_LEN, return_tensors="pt")
        q_emb = model.encode(tokens["input_ids"].to(device), tokens["attention_mask"].to(device))

        # Search
        scores = torch.matmul(q_emb, corpus_embs.T).squeeze(0)
        top100 = torch.topk(scores, 100).indices.tolist()
        retrieved = [str(corpus_df.iloc[idx]["appno"]) for idx in top100]

        recall = len(set(retrieved) & set(gold)) / len(gold)
        recalls.append(recall)

if recalls:
    print(f"\n📊 Quick Evaluation Results (500 test queries):")
    print(f"  Recall@100: {np.mean(recalls)*100:.2f}%")
    print(f"  Expected: 28-34% for paper specification")

    if np.mean(recalls) >= 0.28:
        print("  ✅ EXCELLENT! Matches paper baseline!")
    elif np.mean(recalls) >= 0.24:
        print("  ✅ GOOD! Close to paper baseline.")
    else:
        print("  ⚠️ Below expected. Consider more epochs or checking data.")

print("\n" + "="*60)
print("✅ ALL DONE!")
print("="*60)